# Exercise 1: image folder → JSON → DataFrame

## Tasks

1. **Look:** see the images and decide which fields to extract.
2. **Extract:** ask the model for one JSON record per image.
3. **Collect:** put all records into one DataFrame.
4. **Evaluate:** compare with the hand labels, field by field.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
import base64
import json
import mimetypes
from pathlib import Path
from typing import Literal

import jiwer
import pandas as pd
from openai import OpenAI
from PIL import Image, ImageOps
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

client = OpenAI(max_retries=5, timeout=60)  # reads OPENAI_API_KEY, retries if the API is busy
MODEL = "gpt-6-luna"

DATA = Path("..") / "data"  # the course images and videos

## 1. Look

In [ ]:
folder = DATA / "images"
paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
print(len(paths), "images")

display(ImageOps.contain(Image.open(paths[0]), (400, 300)))  # shrink for display

The fields to extract, written as a Python class. The API must answer in exactly this format.

In [ ]:
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]
    people_count: int
    text_in_image: str | None  # None = no text visible

In [ ]:
PROMPT = (
    "Describe this image. Pick the scene from the list. "
    "Name up to five prominent objects as lowercase singular nouns. "
    "Count the visible people. "
    "Copy the most prominent text. Use null if there is no text "
    "and [unreadable] if the text cannot be read."
)

## 2. One API call, step by step

**Step 1:** the API request is text, so the image bytes are converted to text (base64).

In [ ]:
path = paths[0]
image_text = base64.b64encode(path.read_bytes()).decode()
print(len(image_text), "characters:", image_text[:60], "...")

**Step 2:** one message with the prompt and the image.

In [ ]:
image_type = mimetypes.guess_type(path)[0]  # "image/jpeg" or "image/png"

message = {
    "role": "user",
    "content": [
        {"type": "input_text", "text": PROMPT},
        {"type": "input_image", "image_url": f"data:{image_type};base64,{image_text}", "detail": "low"},
    ],
}

**Step 3:** send the message. `text_format` tells the API to answer in the `ImageRecord` format.

In [ ]:
response = client.responses.parse(
    model=MODEL,
    input=[message],
    text_format=ImageRecord,
    reasoning={"effort": "none"},  # no extra thinking: faster and cheaper
    max_output_tokens=500,
    store=False,  # OpenAI does not keep the answer
)
print(response.output_text)  # the raw answer: JSON text

**Step 4:** the same answer, already checked and converted into an `ImageRecord`.

In [ ]:
answer = response.output_parsed
print(answer.scene)
print(answer.model_dump())  # as a Python dict

**Step 5:** cost. The image is read as tokens, so every image costs money. Estimate before you run: cost per image × number of images.

In [ ]:
PRICE_INPUT = 0.10   # US$ per 1 million input tokens
PRICE_OUTPUT = 0.50  # US$ per 1 million output tokens

usage = response.usage
cost = (usage.input_tokens * PRICE_INPUT + usage.output_tokens * PRICE_OUTPUT) / 1_000_000
print(usage.input_tokens, "input tokens,", usage.output_tokens, "output tokens")
print(f"This image: US$ {cost:.5f}")
print(f"All {len(paths)} images: about US$ {cost * len(paths):.4f}")

**Step 6:** log prompt, model and raw answer, so you can check every result later.

In [ ]:
entry = {"prompt": PROMPT, "model": MODEL, "answer": response.output_text}
with open("api_log.jsonl", "a", encoding="utf-8") as log:
    log.write(json.dumps(entry, ensure_ascii=False) + "\n")

## 3. Put the steps into functions

We need these steps for every image, so we put them into two functions – the same code as above.

In [ ]:
def image_to_input(path):
    """Turn an image file into the text format the API expects."""
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


def ask(prompt, answer_format, images):
    """Send a prompt and a list of images. Returns the answer as a dict."""
    # 1. Put the text and the images into one message.
    content = [{"type": "input_text", "text": prompt}]
    for path in images:
        content.append(image_to_input(path))

    # 2. Send the message. The answer must follow answer_format.
    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        text_format=answer_format,
        reasoning={"effort": "none"},
        max_output_tokens=500,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Return the answer as a Python dict.
    return response.output_parsed.model_dump()

## 4. Collect

The same prompt for every image. We add the filename ourselves – the model does not need to know it.

In [ ]:
rows = []
for path in paths:
    row = ask(PROMPT, ImageRecord, [path])
    row["file"] = path.name
    rows.append(row)

extracted = pd.DataFrame(rows)
extracted.to_json("extracted.jsonl", orient="records", lines=True)
extracted

## 5. Evaluate

Measure **each field separately** – the average hides the weak field.

In [ ]:
# Hand labels: one row per image. Objects are separated by ";".
labels = pd.read_csv(folder / "labels.csv", keep_default_na=False)

# Put prediction and label side by side, matched by filename.
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))

print("Scene accuracy:       ", accuracy_score(scores["scene_ref"], scores["scene_pred"]))
print("People count accuracy:", accuracy_score(scores["people_count_ref"], scores["people_count_pred"]))

**Objects** are a list, so an exact match is too strict. F1 combines
- *precision*: how many of the named objects are correct?
- *recall*: how many of the true objects were found?

`MultiLabelBinarizer` turns each list into a row of 0s and 1s, one column per object name.

In [ ]:
pred = scores["objects_pred"]  # lists from the model
ref = scores["objects_ref"].str.split(";")  # "boat;water" -> ["boat", "water"]

encoder = MultiLabelBinarizer()
encoder.fit(pd.concat([pred, ref]))  # learn all object names
y_pred = encoder.transform(pred)
y_ref = encoder.transform(ref)

print("Mean object F1:", round(f1_score(y_ref, y_pred, average="samples"), 2))

In [ ]:
# F1 for each image on its own.
f1_per_image = []
for i in range(len(scores)):
    f1_per_image.append(f1_score(y_ref[i], y_pred[i], zero_division=0))
scores["objects_f1"] = f1_per_image

**Text** is scored with the character error rate (CER): how many characters must be changed, per character of the reference? Lower is better. Only images with readable reference text count.

In [ ]:
with_text = scores[~scores["text_in_image_ref"].isin(["null", "[unreadable]"])]

reference = list(with_text["text_in_image_ref"])
prediction = list(with_text["text_in_image_pred"].fillna(""))  # no answer = empty text
print("Model:", prediction, "| Label:", reference)
print("CER:", round(jiwer.cer(reference, prediction), 2))

## 6. Look at the worst case

Why can a perfectly valid JSON answer still be wrong?

In [ ]:
worst = scores.loc[scores["objects_f1"].idxmin()]
print(worst["file"], "| F1:", round(worst["objects_f1"], 2))
print("Model:", worst["objects_pred"])
print("Label:", worst["objects_ref"])
display(ImageOps.contain(Image.open(folder / worst["file"]), (400, 300)))

### Jupyter notebook --footer info--

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')